In [7]:
# 1. 清理Colab預設可能衝突的舊版套件
!apt-get remove chromium-browser chromium-chromedriver -y
!apt-get autoremove -y

# 2. 下載並安裝官方Google Chrome穩定版
!wget https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt install -y ./google-chrome-stable_current_amd64.deb
!rm google-chrome-stable_current_amd64.deb

# 3. 安裝Linux虛擬螢幕套件
!apt-get install -y xvfb
!curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash -
!apt-get install -y nodejs
!apt-get install -y ffmpeg

# 4. 一次裝齊所有需要的Python爬蟲套件
!pip install undetected-chromedriver pyvirtualdisplay selenium beautifulsoup4 requests pandas
!pip install -U --pre "yt-dlp[default]"

!curl -fsSL https://deno.land/x/install/install.sh | sh


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Package 'chromium-browser' is not installed, so not removed
Package 'chromium-chromedriver' is not installed, so not removed
0 upgraded, 0 newly installed, 0 to remove and 90 not upgraded.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
0 upgraded, 0 newly installed, 0 to remove and 90 not upgraded.
--2026-07-13 16:24:46--  https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
Resolving dl.google.com (dl.google.com)... 173.194.210.136, 173.194.210.190, 173.194.210.91, ...
Connecting to dl.google.com (dl.google.com)|173.194.210.136|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 133840516 (128M) [application/x-debian-package]
Saving to: ‘google-chrome-stable_current_amd64.deb’

google-chrome-stabl 100%[===================>] 127.64M   288MB/s    in 0.4s    

2026-07-13 16:24:46 (288 MB/s) - ‘google-

In [2]:
#連線測試
import undetected_chromedriver as uc
from pyvirtualdisplay import Display
import time

# 1. 啟動虛擬螢幕
print("正在架設虛擬顯示器...")
display = Display(visible=0, size=(1920, 1080))
display.start()

# 2. 設定啟動參數
options = uc.ChromeOptions()
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
options.add_argument('--disable-gpu')
options.add_argument('--window-size=1920,1080')

options.binary_location = "/usr/bin/google-chrome"

print("啟動Chrome...")
driver = uc.Chrome(options=options, version_main=148)

print("嘗試連線至嘖嘖...")
driver.get("https://www.zeczec.com/")

# 給Cloudflare更充裕的時間跑完JavaScript
time.sleep(8)

print("目前網頁標題：", driver.title)

# 測試完關閉資源
driver.quit()
display.stop()

正在架設虛擬顯示器...
啟動Chrome...


FileNotFoundError: 
---------------------
Could not determine browser executable.
---------------------
Make sure your browser is installed in the default location (path).
If you are sure about the browser executable, you can specify it using
the `browser_executable_path='/path/to/browser/executable` parameter.



In [ ]:
#爬蟲跑到一半當機，或手動按了停止鈕後，用來釋放RAM的
!pkill -f chrome

In [10]:
import os
import re
import time
import csv
import requests
import pandas as pd
import yt_dlp
import subprocess
from bs4 import BeautifulSoup
import undetected_chromedriver as uc
from pyvirtualdisplay import Display
from selenium.webdriver.common.by import By
from google.colab import drive
import shutil

if os.path.exists('/root/.deno/bin'):
    os.environ['PATH'] = '/root/.deno/bin:' + os.environ.get('PATH', '')

# ==========================================
# 新增功能區：1. 專案自動編號  2. FAQ 收集器  3. 專案日期抓取
# ==========================================
# 1.自動編號
category_prefix_map = {
    "出版": "P", "時尚": "F", "設計": "D", "科技": "T",
    "教育": "E", "遊戲": "G", "飲食": "FD", "社會": "S"
}
status_prefix_map = {"成功": "S", "失敗": "F"}
project_counters = {}

def generate_project_code(category_name, status):
    """產生如 GF1 (遊戲+失敗+第1號) 的編號"""
    cat_p = category_prefix_map.get(category_name, "X")
    stat_p = status_prefix_map.get(status, "U")
    prefix = f"{cat_p}{stat_p}"

    if prefix not in project_counters:
        project_counters[prefix] = 1
    else:
        project_counters[prefix] += 1
    return f"{prefix}{project_counters[prefix]}"

# ==========================================
# 1.5 斷點續傳掃描器 (掃描雲端硬碟已完成的專案)
# ==========================================
def load_resume_state(base_dir):
    """掃描已建立的資料夾，還原爬蟲進度與專案編號"""
    scraped_slugs = set()
    category_counts = {}

    global project_counters
    project_counters = {} # 初始化並重新計算編號

    if not os.path.exists(base_dir):
        return scraped_slugs, category_counts

    for main_name in MAIN_TYPES.keys():
        for sub_name in SUB_CATEGORIES.keys():
            for status in ['成功', '失敗']:
                status_dir = os.path.join(base_dir, main_name, sub_name, status)
                if os.path.exists(status_dir):
                    # 抓出該狀態下的所有專案資料夾
                    folders = [f for f in os.listdir(status_dir) if os.path.isdir(os.path.join(status_dir, f)) and "_" in f]
                    category_counts[(main_name, sub_name, status)] = len(folders)

                    for f in folders:
                        # 解析檔名，例如 GF1_rogerems
                        try:
                            proj_code, slug = f.split("_", 1)
                            scraped_slugs.add(slug)

                            # 讓自動編號 (GF1, GS2) 接續目前的數字
                            match = re.match(r'([A-Z]+)(\d+)', proj_code)
                            if match:
                                prefix, num = match.group(1), int(match.group(2))
                                if prefix not in project_counters or num > project_counters[prefix]:
                                    project_counters[prefix] = num
                        except:
                            pass

    return scraped_slugs, category_counts

# 2.FAQ收集
def process_faq_data(driver, project_url, proj_code, save_folder_path, funding_days=30):
    """抓取 FAQ 並回傳 (總數, 更新頻率)"""
    try:
        # 1. 進入 FAQ 專屬網址 (嘖嘖支援直接用網址切換標籤)
        driver.get(f"{project_url}/faqs")
        time.sleep(3) # 【關鍵】必須等待標籤切換與動態框框渲染完畢

        total_count = 0
        tabs = driver.find_elements(By.XPATH, "//*[contains(text(), '常見問答')]")
        for tab in tabs:
            # 用正則表達式把數字抓出來
            match = re.search(r'常見問答\s*(\d+)', tab.text)
            if match:
                total_count = int(match.group(1))
                break

        # 算出頻率
        update_freq = round(total_count / funding_days, 4) if funding_days > 0 else 0

        # 3. 抓取底下的問答文字
        if total_count > 0:
            soup = BeautifulSoup(driver.page_source, 'html.parser')
            faq_text_content = f"專案編號：{proj_code}\nFAQ 總體數：{total_count}\n更新頻率：{update_freq}\n" + "="*30 + "\n\n"

            # 利用你截圖中的第二個特徵：每個問答都會有「更新於 YYYY/MM/DD」
            # 我們找到這些時間標籤，然後往上找它的父容器(也就是那個灰底的框框)
            date_texts = soup.find_all(string=re.compile(r'更新於\s*\d{4}'))

            if date_texts:
                for i, date_text in enumerate(date_texts, 1):
                    current_node = date_text.parent
                    while current_node.parent and len(current_node.parent.find_all(string=re.compile(r'更新於\s*\d{4}'))) == 1:
                        current_node = current_node.parent

                    if current_node:
                        text = current_node.get_text(separator='\n', strip=True)
                        faq_text_content += f"【第 {i} 題】\n{text}\n\n"

            else:
                # 備用方案：如果找不到「更新於」，就抓取整個網頁的主要內容文字
                faq_text_content += soup.get_text(separator='\n', strip=True)[:2000] # 只抓前2000字避免過長

            # 存成 TXT
            txt_file_path = os.path.join(save_folder_path, f"{proj_code}_FAQ.txt")
            with open(txt_file_path, "w", encoding="utf-8") as f:
                f.write(faq_text_content)

        return total_count, update_freq
    except Exception as e:
        print(f"FAQ抓取失敗: {e}")
        return 0, 0

# === [新增] 3. 專案日期抓取 ===
def get_project_dates(page_source):
    """抓取專案的開始與結束日期"""
    start_date_str = ""
    end_date_str = ""
    try:
        soup = BeautifulSoup(page_source, 'html.parser')
        # 尋找包含募資期間的文字，格式通常為 "2023/01/01 12:00 – 2023/02/01 12:00"
        date_text_div = soup.find(string=re.compile(r'\d{4}/\d{2}/\d{2}.+–.+\d{4}/\d{2}/\d{2}'))

        if date_text_div:
            # 切分出兩個符合 YYYY/MM/DD 格式的日期
            dates = re.findall(r'\d{4}/\d{2}/\d{2}', date_text_div)
            if len(dates) >= 2:
                # 將 "YYYY/MM/DD" 轉換為 "YYYY-MM-DD" 以利後續資料處理
                start_date_str = dates[0].replace('/', '-')
                end_date_str = dates[1].replace('/', '-')
    except Exception as e:
        print(f"日期抓取失敗: {e}")

    return start_date_str, end_date_str

# ==========================================
# 0. 系統環境修正 (解決 No supported JavaScript runtime)
# ==========================================
# 在 Colab 執行時，請先跑這行： !apt-get install -y nodejs

# ==========================================
# 1. 核心功能：多媒體下載器 (你的邏輯)
# ==========================================
def zeczec_multimodal_downloader(url, project_name, driver, base_folder_path, proj_code, valid_img_urls):
    # 建立圖片與影片子目錄 (這裡加上了 proj_code 前綴，例如 GF1_images)
    img_path = os.path.join(base_folder_path, f"{proj_code}_images")
    video_path = os.path.join(base_folder_path, f"{proj_code}_videos")
    os.makedirs(img_path, exist_ok=True)
    os.makedirs(video_path, exist_ok=True)

    print(f" [影音任務] 正在掃描：{project_name}")

    driver.get(url)
    time.sleep(3)

    # 模擬捲動觸發 Lazy Load
    for i in range(3):
        driver.execute_script(f"window.scrollTo(0, {i * 1200});")
        time.sleep(1.2)

    resource_log = []

    # --- [A. 圖片抓取] ---
    print("正在掃描圖片...")
    print(f"準備下載 {len(valid_img_urls)} 張內容圖片...")
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}

    # 💡 直接跑迴圈處理傳進來的清單，idx 會自動從 0 開始，完美對齊文本裡的 [img_0]
    for idx, img_url in enumerate(valid_img_urls):
        try:
            resp = requests.get(img_url, headers=headers, timeout=10)
            if resp.status_code == 200:
                ctype = resp.headers.get('Content-Type', '').lower()
                ext = ".gif" if 'gif' in ctype else ".png" if 'png' in ctype else ".webp" if 'webp' in ctype else ".jpg"

                # 檔名強制使用 idx！就算前面的圖失敗了，後面的檔名序號也不會亂掉
                f_name = f"img_{idx}{ext}"
                f_path = os.path.join(img_path, f_name)

                with open(f_path, 'wb') as f:
                    f.write(resp.content)

                # 檔案太小就砍掉 (文本裡會留下一個空包彈標籤 [img_X]，但在機器學習處理上這不會引發錯位，非常安全)
                if os.path.getsize(f_path) > 5000:
                    resource_log.append({"FileName": f_name, "Type": "Image", "URL": img_url})
                else:
                    os.remove(f_path)
        except Exception as e:
            print(f"圖片 img_{idx} 下載失敗: {e}")
            continue

    # --- [B. 影片偵測 - noscript 專攻版] ---
    video_urls = set()
    # 1. 掃描 iframe
    try:
        for iframe in driver.find_elements(By.TAG_NAME, "iframe"):
            v_src = iframe.get_attribute("src") or iframe.get_attribute("data-src")
            if v_src and "youtube.com" in v_src:
                video_urls.add(v_src.split('?')[0].replace("embed/", "watch?v="))
    except: pass

    # 2. 掃描 noscript
    try:
        for ns in driver.find_elements(By.TAG_NAME, "noscript"):
            ns_content = ns.get_attribute("innerHTML")
            if "youtube.com" in ns_content and "embed" in ns_content:
                match = re.search(r'embed/([a-zA-Z0-9_-]{11})', ns_content)
                if match:
                    video_urls.add(f"https://www.youtube.com/watch?v={match.group(1)}")
    except: pass

    # --- [C. 影片下載] ---
    #cookie_file_path = '/content/www.youtube.com_cookies.txt'
    #cookie_file_path = os.path.join(BASE_DIR, 'www.youtube.com_cookies.txt')
    cookie_file_path = '/content/drive/MyDrive/ZecZec_Group_Data/www.youtube.com_cookies.txt'

    if not os.path.exists(cookie_file_path):
      print(f"⚠️ 找不到 cookie 檔案：{cookie_file_path}，將以未登入狀態嘗試下載")
      cookie_file_path = None  # 傳 None 給 yt_dlp 代表不使用 cookie

    if video_urls:
        ydl_opts = {
              'format': 'bestvideo+bestaudio/best',   # 比單純 'best' 更寬鬆，允許影音分開合併
              'outtmpl': f'{video_path}/video_%(title).30s_%(id)s.%(ext)s',
              'windowsfilenames': True,
              'quiet': True,
              # 'verbose': True,
              'no_warnings': True,
              'ignoreerrors': True,
              'sleep_interval': 8,
              'max_sleep_interval': 20,
              'cookiefile': cookie_file_path,
              'js_runtimes': {'deno': {}},
              'extractor_args': {
                  'youtube': {'player_client': ['android', 'mweb', 'web_safari', 'ios']}  # 多加幾個備援身份，其中一個失敗會自動換下一個
              },
              'retries': 3
          }
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            for v_url in video_urls:
                try:
                    print(f"下載影片: {v_url}")
                    ydl.download([v_url])
                    resource_log.append({"FileName": v_url, "Type": "Video", "URL": v_url})
                except Exception as e: print(f"影片下載失敗，可能原因:{e}")

    # 儲存下載紀錄 (這裡加上了 proj_code 前綴，例如 GF1_resource_log.csv)
    if resource_log:
        pd.DataFrame(resource_log).to_csv(f"{base_folder_path}/{proj_code}_resource_log.csv", index=False, encoding="utf-8-sig")

# ==========================================
# 2. 環境與初始化設定
# ==========================================
MAIN_TYPES = {'群眾募資': '0', '預購式專案': '1'}
SUB_CATEGORIES = {
    #'出版': '4',
    #'時尚': '7',
    #'設計': '8',
    '科技': '11',
    #'教育': '12',
    #'遊戲': '13',
    #'飲食': '14',
    #'社會': '15'
}

drive.mount('/content/drive')
BASE_DIR = '/content/drive/MyDrive/ZecZec_Group_Data'
#BASE_DIR = './ZecZec_Dataset'

def create_directories():
    if not os.path.exists(BASE_DIR): os.mkdir(BASE_DIR)
    for main_name in MAIN_TYPES.keys():
        main_path = os.path.join(BASE_DIR, main_name)
        if not os.path.exists(main_path): os.mkdir(main_path)
        for sub_name in SUB_CATEGORIES.keys():
            sub_path = os.path.join(main_path, sub_name)
            if not os.path.exists(sub_path): os.mkdir(sub_path)
            for status_name in ['成功', '失敗']:
                status_path = os.path.join(sub_path, status_name)
                if not os.path.exists(status_path): os.mkdir(status_path)
    print("資料夾結構建立完成！")

def ensure_chrome_installed():
    """確認 Chrome 是否存在，不存在就重新安裝"""
    if not os.path.exists("/usr/bin/google-chrome"):
        print("⚠️ 偵測不到 Chrome，重新安裝中...")
        subprocess.run("apt-get remove -y chromium-browser chromium-chromedriver", shell=True)
        subprocess.run("wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb -O /tmp/chrome.deb", shell=True)
        subprocess.run("apt install -y /tmp/chrome.deb", shell=True)
        subprocess.run("apt-get -f install -y", shell=True)  # 修復依賴問題
        if not os.path.exists("/usr/bin/google-chrome"):
            raise RuntimeError("Chrome 安裝失敗，請檢查 apt 是否有錯誤訊息。")
    print("✅ Chrome 已就緒")

def get_chrome_major_version():
    """動態取得目前安裝的 Chrome 主版本號"""
    out = subprocess.check_output(["/usr/bin/google-chrome", "--version"]).decode()
    match = re.search(r'(\d+)\.\d+\.\d+\.\d+', out)
    if not match:
        raise RuntimeError(f"無法解析 Chrome 版本號，原始輸出：{out}")
    return int(match.group(1))

def get_driver():
    ensure_chrome_installed()
    major_version = get_chrome_major_version()
    print(f"啟動Chrome...(偵測到版本 {major_version})")

    display = Display(visible=0, size=(1920, 1080))
    display.start()

    options = uc.ChromeOptions()
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.binary_location = "/usr/bin/google-chrome"

    driver = uc.Chrome(options=options, version_main=major_version)
    return driver, display

# ==========================================
# 3. 核心解析邏輯 (文字抓取)
# ==========================================
def scrape_project_details(driver, project_url, main_name, sub_name):
    driver.get(project_url)
    time.sleep(3)

    # 18歲驗證
    try:
        xpath_query = "//*[self::button or self::a or self::input][contains(., '我已滿 18 歲')]"
        age_btn = driver.find_element(By.XPATH, xpath_query)
        driver.execute_script("arguments[0].click();", age_btn)
        time.sleep(4)
    except: pass

    # 展開計畫內容
    try:
        expand_btn = driver.find_element(By.CSS_SELECTOR, 'button.js-expand-project')
        driver.execute_script("arguments[0].click();", expand_btn)
    except: pass

    soup = BeautifulSoup(driver.page_source, 'html.parser')
    project_id = project_url.split('/')[-1].split('?')[0]

    og_title = soup.find('meta', property='og:title')
    if og_title and og_title.get('content'):
        project_title = og_title['content']
    elif soup.title:
        project_title = soup.title.get_text(strip=True)
    else:
        project_title = project_id

    project_title = project_title.replace("嘖嘖 | ", "").replace(" | 嘖嘖", "").replace("zeczec", "").strip()

    pct_tag = soup.find(class_='js-percentage-raised')
    if not pct_tag: return "ERROR"

    time_left_tag = soup.find(class_='js-time-left')
    if time_left_tag and any(x in time_left_tag.text for x in ['天', '時', '分']):
        return "ACTIVE"

    pct_val = int(re.sub(r'[^\d]', '', pct_tag.text)) if re.sub(r'[^\d]', '', pct_tag.text).isdigit() else 0
    status_name = '成功' if pct_val >= 100 else '失敗'

    # ==========================================
    # 【新增替換區】 多模態圖文順序保留機制
    # 取代了原本短短兩行的 content_text 抓取邏輯
    # ==========================================
    content_div = soup.find('div', id='project_content')
    target_soup = content_div if content_div else soup

    valid_img_urls = []
    img_index = 0 # 統一的流水號 (貫穿主文與方案)

    # --- 第 1 階段：處理「主文區」的圖片 ---
    for img in target_soup.find_all('img'):
        img_url = img.get('data-src') or img.get('src')
        if not img_url: continue

        if img_url.startswith("data:") or any(x in img_url.lower() for x in ['icon', 'logo', 'avatar', '.svg']):
            continue

        valid_img_urls.append(img_url)
        img.insert_after(f"\n[img_{img_index}]\n")
        img_index += 1 # 號碼遞增

    # 抽取主文 (帶有 [img_X])
    content_text = target_soup.get_text(separator='\n', strip=True) if content_div else "無法抓取內文"


    # --- 第 2 階段：處理「方案區」的內容與圖片 ---
    price_tags = soup.find_all('div', class_='text-black font-bold text-xl flex items-center')
    plan_prices = []
    plans_details = []

    for idx, pt in enumerate(price_tags, 1):
        # 擷取價格供 CSV 使用
        price_match = re.search(r'NT\$\s*([\d,]+)', pt.text)
        if price_match:
            plan_prices.append(price_match.group(1).replace(',', ''))

        # 往上找兩層，鎖定整張方案卡片
        card = pt.parent.parent
        if len(card.get_text(strip=True)) < 30:
            card = card.parent

        if card:
            # 💡【關鍵修復】尋找這張方案卡片裡面的圖片！
            for img in card.find_all('img'):
                img_url = img.get('data-src') or img.get('src')
                if not img_url: continue

                if img_url.startswith("data:") or any(x in img_url.lower() for x in ['icon', 'logo', 'avatar', '.svg']):
                    continue

                # 把方案的圖片也加進同一個下載清單
                valid_img_urls.append(img_url)
                # 延續上面的 img_index 繼續安插標籤
                img.insert_after(f"\n[img_{img_index}]\n")
                img_index += 1

            # 抽文字 (此時如果方案有附圖，就會自動帶有 [img_X] 標籤了！)
            card_text = card.get_text(separator='\n', strip=True)
            plans_details.append(f"▼ 【方案 {idx}】 ▼\n{card_text}")

    prices_str = " | ".join(plan_prices) if plan_prices else "無價格資訊"
    plans_text_output = "\n\n----------------------------------------\n\n".join(plans_details) if plans_details else "無方案資訊"
    # ==========================================

    price_tags = soup.find_all('div', class_='text-black font-bold text-xl flex items-center')
    plan_prices = [re.search(r'NT\$\s*([\d,]+)', t.text).group(1).replace(',', '') for t in price_tags if re.search(r'NT\$\s*([\d,]+)', t.text)]
    prices_str = " | ".join(plan_prices) if plan_prices else "無價格資訊"

    # ==========================================
    # 抓取右側方案詳細內容 (價格、名稱、內容物)
    # ==========================================
    plans_details = []
    for idx, pt in enumerate(price_tags, 1):
        # 順藤摸瓜：往上找兩層，通常這就是包裹整個方案卡片的 <div> 容器
        card = pt.parent.parent

        # 防呆機制：如果抓到的字數太少(代表容器可能包得不夠深)，就再往上一層找
        if len(card.get_text(strip=True)) < 30:
            card = card.parent

        if card:
            # 將卡片內的所有文字抽出來，並且優雅地用換行符號隔開
            card_text = card.get_text(separator='\n', strip=True)
            plans_details.append(f"▼ 【方案 {idx}】 ▼\n{card_text}")

    # 把所有方案組合成一個大字串，中間用虛線隔開
    plans_text_output = "\n\n----------------------------------------\n\n".join(plans_details) if plans_details else "無方案資訊"

    # === 計算實際募資天數 ===
    actual_days = 30 # 預設安全值

    try:
        date_text_div = soup.find(string=re.compile(r'\d{4}/\d{2}/\d{2}.+–.+\d{4}/\d{2}/\d{2}'))

        if date_text_div:
            # 用正則表達式把兩個日期時間抓出來
            dates = re.findall(r'\d{4}/\d{2}/\d{2} \d{2}:\d{2}', date_text_div)
            if len(dates) == 2:
                # 轉換為 datetime 物件來計算時間差
                from datetime import datetime
                start_date_dt = datetime.strptime(dates[0], "%Y/%m/%d %H:%M")
                end_date_dt = datetime.strptime(dates[1], "%Y/%m/%d %H:%M")

                # 計算相差天數，如果算出來是 0 天(例如當天結束)，至少算 1 天避免除以零
                delta_days = (end_date_dt - start_date_dt).days
                actual_days = max(1, delta_days)
    except Exception as e:
        print(f"天數計算失敗，使用預設值 30 天: {e}")
        actual_days = 30

    return {
        "status": status_name,
        "project_id": project_id,
        "project_title": project_title,
        "content_text": content_text,
        "plans_text": plans_text_output,
        "valid_img_urls": valid_img_urls, # 【關鍵新增】把清單往外傳遞給主迴圈！
        "actual_days": actual_days,
        "pct_val": pct_val,
        "csv_row": [main_name, sub_name, status_name, project_id, prices_str, len(plan_prices)]
    }


# ==========================================
# 5. 主爬蟲流程 (含崩潰自動重啟與斷點續傳機制)
# ==========================================
def main_crawler(target_per_status=16):
    create_directories()

    # 1. 讀取雲端硬碟先前的進度
    scraped_slugs, category_counts = load_resume_state(BASE_DIR)
    print(f"\n已讀取中斷點：發現 {len(scraped_slugs)} 個已完成的專案，將自動跳過並接續編號。")

    driver, display = get_driver()

    csv_filename = os.path.join(BASE_DIR, 'projects_summary.csv')

    # === [修改 1] 初始化 CSV 標題列：打開寫完立刻關閉 ===
    if not os.path.exists(csv_filename):
        with open(csv_filename, 'w', newline='', encoding='utf-8') as csvfile:
            writer = csv.writer(csvfile)
            writer.writerow(['主分類', '次分類', '募資狀態', '專案編號', '方案價格列表', '折扣層數', 'FAQ總題數', 'FAQ更新頻率', '開始日期', '結束日期', '總天數', '達標率(%)'])

    for main_name, main_type in MAIN_TYPES.items():
        for sub_name, sub_cat in SUB_CATEGORIES.items():
            success_count = category_counts.get((main_name, sub_name, '成功'), 0)
            failed_count = category_counts.get((main_name, sub_name, '失敗'), 0)
            page_num = 1

            while success_count < target_per_status or failed_count < target_per_status:
                print(f"\n分類: {sub_name} 第 {page_num} 頁 (目前進度: 成功 {success_count}/{target_per_status}, 失敗 {failed_count}/{target_per_status})...")
                list_url = f"https://www.zeczec.com/categories?category={sub_cat}&type={main_type}&page={page_num}"

                try:
                    driver.get(list_url)
                except:
                    print("瀏覽器連線中斷，正在重啟...")
                    driver.quit(); display.stop()
                    driver, display = get_driver()
                    continue

                time.sleep(3)
                soup = BeautifulSoup(driver.page_source, 'html.parser')

                # === [門口過濾機制] 阻擋預熱案 ===
                all_cards = soup.select('a[href^="/projects/"]')
                project_urls = []
                for card in all_cards:
                    href = card.get('href')
                    if 'comments' in href or 'updates' in href: continue

                    card_text = card.get_text(strip=True)
                    if any(keyword in card_text for keyword in ['專案問卷調查中', '即將啟動', '敬請期待', '專案準備中', '即將開始']):
                        print(f"  [略過預熱專案]: {href.split('/')[-1]}")
                        continue

                    project_urls.append(href)
                project_urls = list(dict.fromkeys(project_urls))

                if not project_urls: break

                for url in project_urls:
                    if success_count >= target_per_status and failed_count >= target_per_status: break

                    project_slug = url.split('/')[-1].split('?')[0]

                    if project_slug in scraped_slugs:
                        continue

                    full_url = "https://www.zeczec.com" + url

                    try:
                        result = scrape_project_details(driver, full_url, main_name, sub_name)
                    except Exception as e:
                        print(f"專案解析失敗，嘗試重啟瀏覽器: {e}")
                        driver.quit(); display.stop()
                        driver, display = get_driver()
                        continue

                    if result in ["ACTIVE", "ERROR"]: continue

                    proj_start_date, proj_end_date = get_project_dates(driver.page_source)
                    status = result["status"]

                    if (status == '成功' and success_count >= target_per_status) or \
                       (status == '失敗' and failed_count >= target_per_status): continue

                    proj_code = generate_project_code(sub_name, status)
                    full_folder_name = f"{proj_code}_{project_slug}"

                    project_folder = os.path.join(BASE_DIR, main_name, sub_name, status, full_folder_name)
                    os.makedirs(project_folder, exist_ok=True)

                    content_filename = f"{proj_code}_content.txt"
                    content_filepath = os.path.join(project_folder, content_filename)

                    # 寫入 txt：打開、寫入、關閉
                    with open(content_filepath, 'w', encoding='utf-8') as f:
                        f.write(f"專案名稱：{result['project_title']}\n")
                        f.write("=" * 50 + "\n\n")
                        f.write(result["content_text"])

                    # ==========================================
                    # 獨立寫入 plans.txt (方案詳細資料)
                    # ==========================================
                    plans_filename = f"{proj_code}_plans.txt"
                    plans_filepath = os.path.join(project_folder, plans_filename)

                    with open(plans_filepath, 'w', encoding='utf-8') as f:
                        f.write(f"專案名稱：{result['project_title']} (方案列表)\n")
                        f.write("=" * 50 + "\n\n")
                        f.write(result["plans_text"])

                    funding_duration = result.get("actual_days", 30)
                    faq_count, faq_freq = process_faq_data(driver, full_url, proj_code, project_folder, funding_days=funding_duration)
                    print(f"  ↳ [進度] 總天數: {funding_duration} 天 | 問答總數: {faq_count} 題 | 計算出頻率: {faq_freq} | 期間: {proj_start_date} ~ {proj_end_date}")

                    # === [修改 2] 更新 CSV：每一筆資料都要「打開、附加、立刻關閉」 ===
                    updated_csv_row = result["csv_row"]
                    updated_csv_row[3] = proj_code
                    updated_csv_row.extend([faq_count, faq_freq, proj_start_date, proj_end_date, funding_duration, result["pct_val"]])

                    with open(csv_filename, 'a', newline='', encoding='utf-8') as csvfile:
                        writer = csv.writer(csvfile)
                        writer.writerow(updated_csv_row)
                    # (不需要 flush 了，因為 with 區塊結束時會自動安全存檔並釋放連線)

                    zeczec_multimodal_downloader(full_url, project_slug, driver, project_folder, proj_code, result["valid_img_urls"])

                    if status == '成功': success_count += 1
                    else: failed_count += 1
                    print(f"完成：{project_slug} ({status})")

                page_num += 1

    driver.quit()
    display.stop()
    print("\n所有採集任務已順利結束！")

#執行
if __name__ == "__main__":
    main_crawler(target_per_status=16)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
資料夾結構建立完成！

已讀取中斷點：發現 0 個已完成的專案，將自動跳過並接續編號。
✅ Chrome 已就緒
啟動Chrome...(偵測到版本 150)

分類: 科技 第 1 頁 (目前進度: 成功 0/16, 失敗 0/16)...

分類: 科技 第 2 頁 (目前進度: 成功 0/16, 失敗 0/16)...
  ↳ [進度] 總天數: 34 天 | 問答總數: 15 題 | 計算出頻率: 0.4412 | 期間: 2026-06-02 ~ 2026-07-06
 [影音任務] 正在掃描：zipio
正在掃描圖片...
準備下載 59 張內容圖片...
下載影片: https://www.youtube.com/watch?v=FqAWELkmJT8
完成：zipio (成功)


KeyboardInterrupt: 